# Training Dynamics Analysis
## Comparing Struggling vs Good Folds - Transformer vs CNN

**Objective:** Analyze training/validation loss curves to understand:
1. Why certain subjects perform poorly (struggling folds)
2. Differences between Transformer and CNN convergence
3. Overfitting patterns and generalization gaps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Publication-quality settings
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.family': 'serif'
})

# Paths
ANALYSIS_DIR = Path('/home/abheekp/FusionTransformer/notebooks/analysis')
RESULTS_DIR = ANALYSIS_DIR / 'results_dec22'
FIGURES_DIR = ANALYSIS_DIR / 'figures'
LATEX_FIGURES = Path('/home/abheekp/FusionTransformer/latex_report/figures')
FIGURES_DIR.mkdir(exist_ok=True)
LATEX_FIGURES.mkdir(exist_ok=True)

print(f"Results directory: {RESULTS_DIR}")
print(f"Figures directory: {FIGURES_DIR}")

## 1. Load Training Logs

In [ ]:
# Load transformer training log
trans_log = pd.read_csv(RESULTS_DIR / 'transformer' / 'training_log.csv')
print(f"Transformer log shape: {trans_log.shape}")
print(f"Columns: {trans_log.columns.tolist()}")
print(f"Subjects: {sorted(trans_log['test_subject'].unique())}")

# Load CNN training log if available
cnn_log_path = RESULTS_DIR / 'cnn' / 'training_log.csv'
if cnn_log_path.exists():
    cnn_log = pd.read_csv(cnn_log_path)
    print(f"\nCNN log shape: {cnn_log.shape}")
else:
    cnn_log = None
    print("\nCNN training log not found")

## 2. Load Per-Subject Performance

In [ ]:
# Load scores
trans_scores = pd.read_csv(RESULTS_DIR / 'transformer' / 'scores.csv')
trans_scores = trans_scores[trans_scores['test_subject'] != 'Average'].copy()
trans_scores['test_subject'] = trans_scores['test_subject'].astype(int)

cnn_scores = pd.read_csv(RESULTS_DIR / 'cnn' / 'scores.csv')
cnn_scores = cnn_scores[cnn_scores['test_subject'] != 'Average'].copy()
cnn_scores['test_subject'] = cnn_scores['test_subject'].astype(int)

# Identify struggling and good subjects (based on Transformer F1)
STRUGGLING_SUBJECTS = [31, 38]  # F1 < 80%
GOOD_SUBJECTS = [50, 58]  # F1 > 98%

print("Subject Performance Summary:")
print("="*50)
for s in STRUGGLING_SUBJECTS + GOOD_SUBJECTS:
    trans_f1 = trans_scores[trans_scores['test_subject'] == s]['f1'].values[0]
    cnn_f1 = cnn_scores[cnn_scores['test_subject'] == s]['f1'].values[0]
    category = "STRUGGLING" if s in STRUGGLING_SUBJECTS else "GOOD"
    print(f"S{s} ({category}): Trans={trans_f1:.2f}%, CNN={cnn_f1:.2f}%")

## 3. Extract Training Metrics Per Subject

In [ ]:
def extract_fold_metrics(log_df, subject):
    """Extract train and val metrics for a specific fold/subject"""
    fold_data = log_df[log_df['test_subject'] == subject]
    
    train_data = fold_data[fold_data['phase'] == 'train'].sort_values('epoch')
    val_data = fold_data[fold_data['phase'] == 'val'].sort_values('epoch')
    
    return {
        'train': {
            'epoch': train_data['epoch'].values,
            'loss': train_data['loss'].values,
            'f1': train_data['f1_score'].values,
            'accuracy': train_data['accuracy'].values
        },
        'val': {
            'epoch': val_data['epoch'].values,
            'loss': val_data['loss'].values,
            'f1': val_data['f1_score'].values,
            'accuracy': val_data['accuracy'].values
        }
    }

# Extract metrics for key subjects
struggling_metrics = {s: extract_fold_metrics(trans_log, s) for s in STRUGGLING_SUBJECTS}
good_metrics = {s: extract_fold_metrics(trans_log, s) for s in GOOD_SUBJECTS}

print(f"Epochs per fold: {len(struggling_metrics[31]['train']['epoch'])}")

## 4. Figure 1: Struggling vs Good Folds - Training Dynamics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Colors
colors = {'train': '#1f77b4', 'val': '#ff7f0e'}
struggling_style = {'linestyle': '-', 'alpha': 0.8}
good_style = {'linestyle': '--', 'alpha': 0.8}

# Plot 1: Loss curves - Struggling subjects
ax1 = axes[0, 0]
for s in STRUGGLING_SUBJECTS:
    m = struggling_metrics[s]
    ax1.plot(m['train']['epoch'], m['train']['loss'], label=f'S{s} Train', color=colors['train'], **struggling_style)
    ax1.plot(m['val']['epoch'], m['val']['loss'], label=f'S{s} Val', color=colors['val'], **struggling_style)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Struggling Subjects (F1 < 80%): Loss Curves')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Plot 2: Loss curves - Good subjects
ax2 = axes[0, 1]
for s in GOOD_SUBJECTS:
    m = good_metrics[s]
    ax2.plot(m['train']['epoch'], m['train']['loss'], label=f'S{s} Train', color=colors['train'], **good_style)
    ax2.plot(m['val']['epoch'], m['val']['loss'], label=f'S{s} Val', color=colors['val'], **good_style)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Good Subjects (F1 > 98%): Loss Curves')
ax2.legend(loc='upper right', fontsize=8)
ax2.set_yscale('log')
ax2.grid(True, alpha=0.3)

# Plot 3: F1 curves - Struggling subjects
ax3 = axes[1, 0]
for s in STRUGGLING_SUBJECTS:
    m = struggling_metrics[s]
    ax3.plot(m['train']['epoch'], m['train']['f1'], label=f'S{s} Train', color=colors['train'], **struggling_style)
    ax3.plot(m['val']['epoch'], m['val']['f1'], label=f'S{s} Val', color=colors['val'], **struggling_style)
ax3.set_xlabel('Epoch')
ax3.set_ylabel('F1 Score (%)')
ax3.set_title('Struggling Subjects: F1 Score Curves')
ax3.legend(loc='lower right', fontsize=8)
ax3.grid(True, alpha=0.3)
ax3.set_ylim([60, 100])

# Plot 4: F1 curves - Good subjects
ax4 = axes[1, 1]
for s in GOOD_SUBJECTS:
    m = good_metrics[s]
    ax4.plot(m['train']['epoch'], m['train']['f1'], label=f'S{s} Train', color=colors['train'], **good_style)
    ax4.plot(m['val']['epoch'], m['val']['f1'], label=f'S{s} Val', color=colors['val'], **good_style)
ax4.set_xlabel('Epoch')
ax4.set_ylabel('F1 Score (%)')
ax4.set_title('Good Subjects: F1 Score Curves')
ax4.legend(loc='lower right', fontsize=8)
ax4.grid(True, alpha=0.3)
ax4.set_ylim([60, 100])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'struggling_vs_good_training.png', bbox_inches='tight', dpi=300)
plt.savefig(LATEX_FIGURES / 'struggling_vs_good_training.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"Saved: {FIGURES_DIR / 'struggling_vs_good_training.png'}")

## 5. Compute Convergence Statistics

In [ ]:
def compute_convergence_stats(metrics):
    """Compute convergence statistics for a fold"""
    train_loss = metrics['train']['loss']
    val_loss = metrics['val']['loss']
    train_f1 = metrics['train']['f1']
    val_f1 = metrics['val']['f1']
    
    # Final metrics (last 5 epochs average for stability)
    final_train_loss = np.mean(train_loss[-5:])
    final_val_loss = np.mean(val_loss[-5:])
    final_train_f1 = np.mean(train_f1[-5:])
    final_val_f1 = np.mean(val_f1[-5:])
    
    # Generalization gap
    gen_gap_loss = final_val_loss - final_train_loss
    gen_gap_f1 = final_train_f1 - final_val_f1
    
    # Convergence speed (epochs to reach 90% of final F1)
    target_f1 = final_train_f1 * 0.9
    conv_epoch = np.argmax(train_f1 >= target_f1) + 1 if np.any(train_f1 >= target_f1) else len(train_f1)
    
    # Stability (std of last 10 epochs)
    stability = np.std(val_f1[-10:])
    
    return {
        'final_train_loss': final_train_loss,
        'final_val_loss': final_val_loss,
        'final_train_f1': final_train_f1,
        'final_val_f1': final_val_f1,
        'gen_gap_loss': gen_gap_loss,
        'gen_gap_f1': gen_gap_f1,
        'convergence_epoch': conv_epoch,
        'stability': stability
    }

# Compute stats for all subjects
all_subjects = sorted(trans_log['test_subject'].unique())
all_stats = {}
for s in all_subjects:
    metrics = extract_fold_metrics(trans_log, s)
    all_stats[s] = compute_convergence_stats(metrics)

# Create summary table
stats_df = pd.DataFrame(all_stats).T
stats_df.index.name = 'subject'
stats_df = stats_df.reset_index()

# Add test F1 from scores
stats_df = stats_df.merge(trans_scores[['test_subject', 'f1']], 
                          left_on='subject', right_on='test_subject')
stats_df = stats_df.rename(columns={'f1': 'test_f1'})
stats_df = stats_df.drop('test_subject', axis=1)

print("Convergence Statistics Summary:")
print("="*80)
print(stats_df.round(3).to_string())

## 6. Analyze Struggling vs Good Subjects

In [ ]:
# Categorize subjects
stats_df['category'] = 'Medium'
stats_df.loc[stats_df['test_f1'] < 82, 'category'] = 'Struggling'
stats_df.loc[stats_df['test_f1'] > 95, 'category'] = 'Good'

print("\nSubject Categories:")
print(stats_df.groupby('category').size())

# Compare key metrics
print("\n" + "="*60)
print("KEY FINDING: Struggling vs Good Subjects Analysis")
print("="*60)

struggling = stats_df[stats_df['category'] == 'Struggling']
good = stats_df[stats_df['category'] == 'Good']

metrics_to_compare = ['gen_gap_f1', 'final_val_f1', 'convergence_epoch', 'stability']

for metric in metrics_to_compare:
    str_mean = struggling[metric].mean()
    good_mean = good[metric].mean()
    diff = good_mean - str_mean
    
    # Statistical test
    if len(struggling) >= 2 and len(good) >= 2:
        t_stat, p_val = stats.ttest_ind(struggling[metric], good[metric])
        sig = "*" if p_val < 0.05 else ""
    else:
        p_val = np.nan
        sig = ""
    
    print(f"\n{metric}:")
    print(f"  Struggling: {str_mean:.3f}")
    print(f"  Good:       {good_mean:.3f}")
    print(f"  Difference: {diff:+.3f} (p={p_val:.4f}) {sig}")

## 7. Figure 2: Averaged Training Curves with Confidence Intervals

In [ ]:
def compute_averaged_curves(log_df):
    """Compute averaged training curves across all folds"""
    subjects = sorted(log_df['test_subject'].unique())
    
    # Collect per-fold curves
    train_losses = []
    val_losses = []
    train_f1s = []
    val_f1s = []
    
    for s in subjects:
        fold_data = log_df[log_df['test_subject'] == s]
        train_data = fold_data[fold_data['phase'] == 'train'].sort_values('epoch')
        val_data = fold_data[fold_data['phase'] == 'val'].sort_values('epoch')
        
        train_losses.append(train_data['loss'].values)
        val_losses.append(val_data['loss'].values)
        train_f1s.append(train_data['f1_score'].values)
        val_f1s.append(val_data['f1_score'].values)
    
    # Stack and compute statistics
    train_losses = np.array(train_losses)
    val_losses = np.array(val_losses)
    train_f1s = np.array(train_f1s)
    val_f1s = np.array(val_f1s)
    
    epochs = np.arange(1, train_losses.shape[1] + 1)
    
    return {
        'epochs': epochs,
        'train_loss_mean': np.mean(train_losses, axis=0),
        'train_loss_std': np.std(train_losses, axis=0),
        'val_loss_mean': np.mean(val_losses, axis=0),
        'val_loss_std': np.std(val_losses, axis=0),
        'train_f1_mean': np.mean(train_f1s, axis=0),
        'train_f1_std': np.std(train_f1s, axis=0),
        'val_f1_mean': np.mean(val_f1s, axis=0),
        'val_f1_std': np.std(val_f1s, axis=0)
    }

avg_curves = compute_averaged_curves(trans_log)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss plot
ax1 = axes[0]
epochs = avg_curves['epochs']
ax1.plot(epochs, avg_curves['train_loss_mean'], 'b-', label='Train', linewidth=2)
ax1.fill_between(epochs, 
                  avg_curves['train_loss_mean'] - avg_curves['train_loss_std'],
                  avg_curves['train_loss_mean'] + avg_curves['train_loss_std'],
                  alpha=0.2, color='blue')
ax1.plot(epochs, avg_curves['val_loss_mean'], 'r-', label='Validation', linewidth=2)
ax1.fill_between(epochs,
                  avg_curves['val_loss_mean'] - avg_curves['val_loss_std'],
                  avg_curves['val_loss_mean'] + avg_curves['val_loss_std'],
                  alpha=0.2, color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Transformer: Averaged Loss (21 LOSO Folds)')
ax1.legend()
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# F1 plot
ax2 = axes[1]
ax2.plot(epochs, avg_curves['train_f1_mean'], 'b-', label='Train', linewidth=2)
ax2.fill_between(epochs,
                  avg_curves['train_f1_mean'] - avg_curves['train_f1_std'],
                  avg_curves['train_f1_mean'] + avg_curves['train_f1_std'],
                  alpha=0.2, color='blue')
ax2.plot(epochs, avg_curves['val_f1_mean'], 'r-', label='Validation', linewidth=2)
ax2.fill_between(epochs,
                  avg_curves['val_f1_mean'] - avg_curves['val_f1_std'],
                  avg_curves['val_f1_mean'] + avg_curves['val_f1_std'],
                  alpha=0.2, color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Score (%)')
ax2.set_title('Transformer: Averaged F1 Score (21 LOSO Folds)')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([75, 100])

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'transformer_averaged_training.png', bbox_inches='tight', dpi=300)
plt.savefig(LATEX_FIGURES / 'transformer_averaged_training.png', bbox_inches='tight', dpi=300)
plt.show()

# Print final statistics
print(f"\nFinal Epoch Statistics:")
print(f"  Train Loss: {avg_curves['train_loss_mean'][-1]:.5f} +/- {avg_curves['train_loss_std'][-1]:.5f}")
print(f"  Val Loss:   {avg_curves['val_loss_mean'][-1]:.5f} +/- {avg_curves['val_loss_std'][-1]:.5f}")
print(f"  Train F1:   {avg_curves['train_f1_mean'][-1]:.2f}% +/- {avg_curves['train_f1_std'][-1]:.2f}%")
print(f"  Val F1:     {avg_curves['val_f1_mean'][-1]:.2f}% +/- {avg_curves['val_f1_std'][-1]:.2f}%")

## 8. Transformer vs CNN Training Comparison

In [ ]:
if cnn_log is not None:
    cnn_avg_curves = compute_averaged_curves(cnn_log)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Loss comparison
    ax1 = axes[0]
    ax1.plot(avg_curves['epochs'], avg_curves['val_loss_mean'], 'b-', 
             label='Transformer', linewidth=2)
    ax1.fill_between(avg_curves['epochs'],
                      avg_curves['val_loss_mean'] - avg_curves['val_loss_std'],
                      avg_curves['val_loss_mean'] + avg_curves['val_loss_std'],
                      alpha=0.2, color='blue')
    ax1.plot(cnn_avg_curves['epochs'], cnn_avg_curves['val_loss_mean'], 'r-', 
             label='CNN', linewidth=2)
    ax1.fill_between(cnn_avg_curves['epochs'],
                      cnn_avg_curves['val_loss_mean'] - cnn_avg_curves['val_loss_std'],
                      cnn_avg_curves['val_loss_mean'] + cnn_avg_curves['val_loss_std'],
                      alpha=0.2, color='red')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Validation Loss')
    ax1.set_title('Transformer vs CNN: Validation Loss')
    ax1.legend()
    ax1.set_yscale('log')
    ax1.grid(True, alpha=0.3)
    
    # F1 comparison
    ax2 = axes[1]
    ax2.plot(avg_curves['epochs'], avg_curves['val_f1_mean'], 'b-',
             label='Transformer', linewidth=2)
    ax2.fill_between(avg_curves['epochs'],
                      avg_curves['val_f1_mean'] - avg_curves['val_f1_std'],
                      avg_curves['val_f1_mean'] + avg_curves['val_f1_std'],
                      alpha=0.2, color='blue')
    ax2.plot(cnn_avg_curves['epochs'], cnn_avg_curves['val_f1_mean'], 'r-',
             label='CNN', linewidth=2)
    ax2.fill_between(cnn_avg_curves['epochs'],
                      cnn_avg_curves['val_f1_mean'] - cnn_avg_curves['val_f1_std'],
                      cnn_avg_curves['val_f1_mean'] + cnn_avg_curves['val_f1_std'],
                      alpha=0.2, color='red')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Validation F1 Score (%)')
    ax2.set_title('Transformer vs CNN: Validation F1')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'transformer_vs_cnn_training.png', bbox_inches='tight', dpi=300)
    plt.savefig(LATEX_FIGURES / 'transformer_vs_cnn_training.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print("CNN training log not available for comparison")

## 9. Figure 3: Generalization Gap Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: Val F1 vs Test F1 (generalization)
ax1 = axes[0]
colors_cat = {'Struggling': 'red', 'Medium': 'gray', 'Good': 'green'}
for cat in ['Struggling', 'Medium', 'Good']:
    cat_data = stats_df[stats_df['category'] == cat]
    ax1.scatter(cat_data['final_val_f1'], cat_data['test_f1'], 
                c=colors_cat[cat], label=cat, s=100, alpha=0.7)

# Add diagonal line (perfect generalization)
lims = [70, 100]
ax1.plot(lims, lims, 'k--', alpha=0.5, label='Perfect generalization')
ax1.set_xlim(lims)
ax1.set_ylim(lims)
ax1.set_xlabel('Validation F1 (%)')
ax1.set_ylabel('Test F1 (%)')
ax1.set_title('Validation vs Test F1 (Generalization Gap)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Add annotations for key subjects
for s in STRUGGLING_SUBJECTS + GOOD_SUBJECTS:
    row = stats_df[stats_df['subject'] == s].iloc[0]
    ax1.annotate(f'S{s}', (row['final_val_f1'], row['test_f1']), 
                 xytext=(5, 5), textcoords='offset points', fontsize=8)

# Histogram: Generalization gap distribution
ax2 = axes[1]
gap = stats_df['final_val_f1'] - stats_df['test_f1']
ax2.hist(gap, bins=15, edgecolor='black', alpha=0.7)
ax2.axvline(gap.mean(), color='red', linestyle='--', label=f'Mean: {gap.mean():.1f}%')
ax2.set_xlabel('Val-Test F1 Gap (%)')
ax2.set_ylabel('Number of Subjects')
ax2.set_title('Distribution of Generalization Gap')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'generalization_gap_analysis.png', bbox_inches='tight', dpi=300)
plt.savefig(LATEX_FIGURES / 'generalization_gap_analysis.png', bbox_inches='tight', dpi=300)
plt.show()

print(f"\nGeneralization Gap Statistics:")
print(f"  Mean:   {gap.mean():.2f}%")
print(f"  Std:    {gap.std():.2f}%")
print(f"  Min:    {gap.min():.2f}% (S{stats_df.loc[gap.idxmin(), 'subject']})")
print(f"  Max:    {gap.max():.2f}% (S{stats_df.loc[gap.idxmax(), 'subject']})")

## 10. Statistical Summary Table

In [ ]:
# Create summary table for paper
summary = stats_df.groupby('category').agg({
    'final_train_f1': ['mean', 'std'],
    'final_val_f1': ['mean', 'std'],
    'test_f1': ['mean', 'std'],
    'gen_gap_f1': ['mean', 'std'],
    'convergence_epoch': ['mean', 'std']
}).round(2)

summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
print("\nTraining Dynamics Summary by Subject Category:")
print("="*80)
print(summary.to_string())

# Save as CSV
summary.to_csv(ANALYSIS_DIR / 'tables' / 'training_dynamics_summary.csv')
print(f"\nSaved: {ANALYSIS_DIR / 'tables' / 'training_dynamics_summary.csv'}")

## 11. Key Findings Summary

In [ ]:
print("\n" + "="*70)
print("KEY FINDINGS: Training Dynamics Analysis")
print("="*70)

print("\n1. STRUGGLING SUBJECTS (F1 < 82%):")
print(f"   - Subjects: {stats_df[stats_df['category']=='Struggling']['subject'].tolist()}")
print(f"   - Higher generalization gap: {struggling['gen_gap_f1'].mean():.2f}% vs {good['gen_gap_f1'].mean():.2f}%")
print(f"   - Lower validation F1: {struggling['final_val_f1'].mean():.2f}% vs {good['final_val_f1'].mean():.2f}%")

print("\n2. GOOD SUBJECTS (F1 > 95%):")
print(f"   - Subjects: {stats_df[stats_df['category']=='Good']['subject'].tolist()}")
print(f"   - Faster convergence: {good['convergence_epoch'].mean():.1f} vs {struggling['convergence_epoch'].mean():.1f} epochs")
print(f"   - More stable: {good['stability'].mean():.3f} vs {struggling['stability'].mean():.3f} (std of last 10 epochs)")

print("\n3. OVERALL TRAINING DYNAMICS:")
print(f"   - Average final train F1: {stats_df['final_train_f1'].mean():.2f}% +/- {stats_df['final_train_f1'].std():.2f}%")
print(f"   - Average final val F1: {stats_df['final_val_f1'].mean():.2f}% +/- {stats_df['final_val_f1'].std():.2f}%")
print(f"   - Average test F1: {stats_df['test_f1'].mean():.2f}% +/- {stats_df['test_f1'].std():.2f}%")
print(f"   - Mean generalization gap (val-test): {gap.mean():.2f}%")

print("\n4. IMPLICATIONS FOR PAPER:")
print("   - Struggling subjects have gentler falls (lower SMV, per signal analysis)")
print("   - These gentler falls are harder to distinguish from ADLs")
print("   - Model converges well but generalizes poorly to these edge cases")
print("   - Suggests need for data augmentation or specialized handling")

## 12. Save All Figures for LaTeX

In [ ]:
import shutil

# List all generated figures
generated_figures = list(FIGURES_DIR.glob('*.png'))
print(f"Generated {len(generated_figures)} figures:")
for fig in generated_figures:
    print(f"  - {fig.name}")
    
# Copy to latex directory
print(f"\nAll figures also saved to: {LATEX_FIGURES}")